In [3]:
import pandas as pd
import numpy as np
import os

In [4]:
output_path = r"C:\Users\DELL\OneDrive\Desktop\supply-chain-operations-analytics\data\intervention_priority_output.csv"

optimization_df = pd.read_csv(output_path)

print("Rows:", len(optimization_df))
print("Columns:", len(optimization_df.columns))

optimization_df.head()

Rows: 99441
Columns: 15


,order_id,customer_state,order_item_count,product_count,seller_count,total_order_value,freight_value,late_probability,risk_score,risk_band,impact_score,priority_score,intervention_tier,recommended_action,is_late
0,00010242fe8c5a6d1ba2dd792cb16214,RJ,1.0,1.0,1.0,72.19,13.29,0.507138,50.713810,High,30.324975,42.558276,Standard Intervention,Monitor and review,0
1,00018f77f2f0320c557190d7a144bdd3,SP,1.0,1.0,1.0,259.83,19.93,0.496976,49.697636,Medium,64.913628,55.784033,High Priority,Prioritize fulfillment monitoring,0
2,000229ec398224ef6ca0657da4fc703e,MG,1.0,1.0,1.0,216.87,17.87,0.496846,49.684551,Medium,56.732248,52.503630,High Priority,Prioritize fulfillment monitoring,0
3,00024acbcdf0a6daa1e931b038114c75,SP,1.0,1.0,1.0,25.78,12.79,0.481683,48.168323,Medium,21.640362,37.557139,Standard Intervention,Monitor and review,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,SP,1.0,1.0,1.0,218.04,18.14,0.496670,49.666972,Medium,57.198468,52.679570,High Priority,Prioritize fulfillment monitoring,0


In [5]:
print(optimization_df.columns.tolist())

['order_id', 'customer_state', 'order_item_count', 'product_count', 'seller_count', 'total_order_value', 'freight_value', 'late_probability', 'risk_score', 'risk_band', 'impact_score', 'priority_score', 'intervention_tier', 'recommended_action', 'is_late']


In [6]:
optimization_df[
    ["risk_score", "priority_score", "intervention_tier"]
].describe(include="all")

,risk_score,priority_score,intervention_tier
count,99441.000000,97917.000000,97917
unique,NaN,NaN,2
top,NaN,NaN,Standard Intervention
freq,NaN,NaN,55176
mean,49.652469,48.813007,NaN
std,1.478039,7.525980,NaN
min,41.414451,32.837591,NaN
25%,48.474893,42.867243,NaN
50%,49.450738,48.584573,NaN
75%,50.754085,54.470377,NaN


In [7]:
optimization_df[
    ["risk_score", "priority_score", "intervention_tier"]
].isnull().sum()

risk_score              0
priority_score       1524
intervention_tier    1524
dtype: int64

In [8]:
print(
    optimization_df["intervention_tier"]
    .value_counts(dropna=False)
)

intervention_tier
Standard Intervention    55176
High Priority            42741
NaN                       1524
Name: count, dtype: int64


In [9]:
print(
    optimization_df["intervention_tier"]
    .value_counts(normalize=True, dropna=False) * 100
)

intervention_tier
Standard Intervention    55.486168
High Priority            42.981265
NaN                       1.532567
Name: proportion, dtype: float64


In [10]:
optimization_df = optimization_df.sort_values(
    by=["priority_score", "risk_score"],
    ascending=False
).reset_index(drop=True)

optimization_df.head(20)

,order_id,customer_state,order_item_count,product_count,seller_count,total_order_value,freight_value,late_probability,risk_score,risk_band,impact_score,priority_score,intervention_tier,recommended_action,is_late
0,1baafdbda29613148a2517059b996b95,RJ,6.0,1.0,1.0,881.52,401.58,0.529111,52.911057,High,99.374759,71.496538,High Priority,Prioritize fulfillment monitoring,0
1,cfed507ac357129f750f05a0d7d71b15,MA,3.0,1.0,1.0,2091.33,711.33,0.526760,52.676013,High,99.604828,71.447539,High Priority,Prioritize fulfillment monitoring,1
2,8339b608be0d84fca9d8da68b58332c3,BA,6.0,1.0,1.0,765.84,165.90,0.529539,52.953936,High,99.111852,71.417103,High Priority,Prioritize fulfillment monitoring,0
3,54d6f9d8f56229d3da815add65ec2408,RJ,2.0,1.0,1.0,348.14,114.16,0.551399,55.139865,High,95.442300,71.260839,High Priority,Prioritize fulfillment monitoring,1
4,3182ecdf3da0331bf56a6b4ff5e069d7,AL,4.0,1.0,1.0,749.64,190.04,0.525873,52.587342,High,98.985466,71.146591,High Priority,Prioritize fulfillment monitoring,1
5,7f8df73aeab603648dec28a5f6cfadc3,BA,4.0,1.0,1.0,1803.24,207.24,0.518336,51.833610,High,99.711045,70.984584,High Priority,Prioritize fulfillment monitoring,0
6,736e1922ae60d0d6a89247b851902527,ES,4.0,1.0,1.0,7274.88,114.88,0.518340,51.833993,High,99.694322,70.978125,High Priority,Prioritize fulfillment monitoring,0
7,03caa2c082116e1d31e67e9ae3700499,RJ,8.0,1.0,1.0,13664.08,224.08,0.516243,51.624264,High,99.970304,70.962680,High Priority,Prioritize fulfillment monitoring,0
8,17784b9fbb37fb0bdc230d8ed6f6b355,BA,6.0,1.0,1.0,1342.98,502.98,0.517425,51.742521,High,99.727464,70.936498,High Priority,Prioritize fulfillment monitoring,0
9,7607b319daef63a1c1a41b1cd6f4a54c,CE,4.0,1.0,1.0,1507.56,191.56,0.515307,51.530722,High,99.644254,70.776135,High Priority,Prioritize fulfillment monitoring,0


In [11]:
INTERVENTION_CAPACITY = 500

print("Available candidates:", len(optimization_df))
print("Intervention capacity:", INTERVENTION_CAPACITY)

Available candidates: 99441
Intervention capacity: 500


In [12]:
selected_df = optimization_df.head(
    INTERVENTION_CAPACITY
).copy()

print("Selected candidates:", len(selected_df))

Selected candidates: 500


In [13]:
# ============================================================
# STEP 6D — INVESTIGATE NULL PRIORITY RECORDS
# ============================================================

null_priority = optimization_df[
    optimization_df["priority_score"].isna()
].copy()

print("NULL priority records:", len(null_priority))

display(
    null_priority[
        [
            "order_id",
            "late_probability",
            "risk_score",
            "risk_band",
            "impact_score",
            "priority_score",
            "intervention_tier",
            "is_late"
        ]
    ].head(20)
)

NULL priority records: 1524


,order_id,late_probability,risk_score,risk_band,impact_score,priority_score,intervention_tier,is_late
97917,6241cda1675cb9e42e0a934f7c71812a,0.536686,53.668610,High,NaN,NaN,NaN,0
97918,af264f3527e94e431f0dcd56cd6b406d,0.533820,53.382019,High,NaN,NaN,NaN,0
97919,a0fdcd775c7cf9f4bfa70207f7d6e239,0.533046,53.304569,High,NaN,NaN,NaN,1
97920,08ee9c1b1da47ffd2a7c7163aa134b16,0.531122,53.112199,High,NaN,NaN,NaN,0
97921,c776863a93dc0740c6e7d78104b21413,0.531050,53.105002,High,NaN,NaN,NaN,1
97922,bb7d0e5b853f82d1aac9d4f51ec59500,0.530909,53.090852,High,NaN,NaN,NaN,1
97923,ee55010a370e987f203499f435a59847,0.530283,53.028282,High,NaN,NaN,NaN,0
97924,f56dc9034c3962e36d49fb73baa12800,0.529928,52.992843,High,NaN,NaN,NaN,0
97925,fa5dc4c3b8326a0da88dc49b20e0ed27,0.529919,52.991928,High,NaN,NaN,NaN,0
97926,91bcbd0102f5188ea47df5bcc83f3cad,0.529761,52.976126,High,NaN,NaN,NaN,0


In [14]:
print("Risk band distribution:")
print(null_priority["risk_band"].value_counts(dropna=False))

print("\nLate status:")
print(null_priority["is_late"].value_counts(dropna=False))

print("\nLate probability:")
print(null_priority["late_probability"].describe())

print("\nImpact score:")
print(null_priority["impact_score"].describe())

Risk band distribution:
risk_band
Medium    982
High      542
Name: count, dtype: int64

Late status:
is_late
0    1359
1     165
Name: count, dtype: int64

Late probability:
count    1524.000000
mean        0.498340
std         0.014041
min         0.440599
25%         0.490518
50%         0.492390
75%         0.511316
max         0.536686
Name: late_probability, dtype: float64

Impact score:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: impact_score, dtype: float64


In [15]:
# Check which input to priority_score contains NULLs

priority_inputs = [
    "late_probability",
    "risk_score",
    "impact_score",
    "total_order_value",
    "freight_value",
    "order_item_count",
    "product_count",
    "seller_count"
]

optimization_df[priority_inputs].isna().sum()

late_probability        0
risk_score              0
impact_score         1524
total_order_value     775
freight_value         775
order_item_count      775
product_count         775
seller_count          775
dtype: int64

In [16]:
optimization_df[
    optimization_df["priority_score"].isna()
][priority_inputs].isna().sum()

late_probability        0
risk_score              0
impact_score         1524
total_order_value     775
freight_value         775
order_item_count      775
product_count         775
seller_count          775
dtype: int64

In [17]:
# ============================================================
# ROOT-CAUSE CHECK: ORDERS WITH MISSING IMPACT INPUTS
# ============================================================

affected = optimization_df[
    optimization_df["impact_score"].isna()
].copy()

print("Affected orders:", len(affected))

print("\nOrder status distribution:")
print(
    affected["order_status"].value_counts(dropna=False)
    if "order_status" in affected.columns
    else "order_status is not available in intervention output"
)

Affected orders: 1524

Order status distribution:
order_status is not available in intervention output


In [18]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

server = r"localhost\SQLEXPRESS"
database = "SupplyChainAnalytics"

connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" +
    quote_plus(connection_string)
)

In [19]:
# ============================================================
# LOAD ORDER-LEVEL ANALYTICS FOR ROOT-CAUSE CHECK
# ============================================================

query = """
SELECT *
FROM dbo.order_analytics
"""

order_analytics = pd.read_sql(query, engine)

print("Order analytics rows:", len(order_analytics))
print("Columns:", len(order_analytics.columns))

Order analytics rows: 99441
Columns: 40


In [20]:
affected_orders = affected[["order_id"]].merge(
    order_analytics,
    on="order_id",
    how="left"
)

print("Affected orders after matching:", len(affected_orders))

Affected orders after matching: 1524


In [21]:
# ============================================================
# STEP 6D — CHECK MISSING METRICS IN AFFECTED ORDERS
# ============================================================

metrics = [
    "order_status",
    "order_item_count",
    "product_count",
    "seller_count",
    "order_item_value",
    "freight_value",
    "total_order_value"
]

print("NULL counts among affected orders:\n")

print(
    affected_orders[metrics].isna().sum()
)

NULL counts among affected orders:

order_status           0
order_item_count     775
product_count        775
seller_count         775
order_item_value     775
freight_value        775
total_order_value    775
dtype: int64


In [22]:
# See the actual affected records

display(
    affected_orders[
        [
            "order_id",
            "order_status",
            "order_item_count",
            "product_count",
            "seller_count",
            "order_item_value",
            "freight_value",
            "total_order_value"
        ]
    ].head(20)
)

,order_id,order_status,order_item_count,product_count,seller_count,order_item_value,freight_value,total_order_value
0,6241cda1675cb9e42e0a934f7c71812a,delivered,1.0,1.0,1.0,10.99,15.10,26.09
1,af264f3527e94e431f0dcd56cd6b406d,unavailable,NaN,NaN,NaN,NaN,NaN,NaN
2,a0fdcd775c7cf9f4bfa70207f7d6e239,delivered,1.0,1.0,1.0,45.00,26.61,71.61
3,08ee9c1b1da47ffd2a7c7163aa134b16,unavailable,NaN,NaN,NaN,NaN,NaN,NaN
4,c776863a93dc0740c6e7d78104b21413,delivered,1.0,1.0,1.0,176.97,87.11,264.08
5,bb7d0e5b853f82d1aac9d4f51ec59500,delivered,1.0,1.0,1.0,148.85,21.84,170.69
6,ee55010a370e987f203499f435a59847,delivered,1.0,1.0,1.0,99.90,18.00,117.90
7,f56dc9034c3962e36d49fb73baa12800,unavailable,NaN,NaN,NaN,NaN,NaN,NaN
8,fa5dc4c3b8326a0da88dc49b20e0ed27,unavailable,NaN,NaN,NaN,NaN,NaN,NaN
9,91bcbd0102f5188ea47df5bcc83f3cad,delivered,1.0,1.0,1.0,209.90,18.00,227.90


In [23]:
# ============================================================
# CHECK STATUS OF AFFECTED ORDERS
# ============================================================

print(
    affected_orders["order_status"]
    .value_counts(dropna=False)
)

order_status
delivered      646
unavailable    603
canceled       181
shipped         76
invoiced         7
processing       6
created          5
Name: count, dtype: int64


In [24]:
# ============================================================
# IMPACT SCORE INPUT CHECK
# ============================================================

affected_orders["missing_economic_data"] = (
    affected_orders["total_order_value"].isna()
    |
    affected_orders["freight_value"].isna()
)

print(
    affected_orders["missing_economic_data"].value_counts()
)

missing_economic_data
True     775
False    749
Name: count, dtype: int64


In [25]:
print(
    affected_orders.groupby("missing_economic_data")[
        ["order_item_count", "product_count", "seller_count",
         "total_order_value", "freight_value"]
    ].apply(lambda x: x.isna().sum())
)

                       order_item_count  product_count  seller_count  \
missing_economic_data                                                  
False                                 0              0             0   
True                                775            775           775   

                       total_order_value  freight_value  
missing_economic_data                                    
False                                  0              0  
True                                 775            775  


In [26]:
# ============================================================
# STEP 6E — VERIFY ORDER ITEM MATCHING
# ============================================================

query = """
SELECT
    o.order_id,
    o.order_status,
    COUNT(oi.order_id) AS order_item_rows,
    SUM(oi.price) AS item_value,
    SUM(oi.freight_value) AS freight_value
FROM dbo.raw_orders o
LEFT JOIN dbo.raw_order_items oi
    ON o.order_id = oi.order_id
WHERE o.order_id IN (
    SELECT order_id
    FROM dbo.order_analytics
    WHERE total_order_value IS NULL
)
GROUP BY
    o.order_id,
    o.order_status
"""

missing_item_check = pd.read_sql(query, engine)

print("Rows checked:", len(missing_item_check))

display(
    missing_item_check["order_status"]
    .value_counts(dropna=False)
)

display(missing_item_check.head(20))

Rows checked: 775


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

,order_id,order_status,order_item_rows,item_value,freight_value
0,00b1cb0320190ca0daa2c88b35206009,canceled,0,None,None
1,00d0ffd14774da775ac832ba8520510f,canceled,0,None,None
2,02e723e8edb4a123d414f56cc9c4665e,canceled,0,None,None
3,032cf17f7a713287e46999fddea6051b,canceled,0,None,None
4,03310aa823a66056268a3bab36e827fb,canceled,0,None,None
5,04a2ba381317648726f46210a46ece95,canceled,0,None,None
6,0623dbbd06c0d4059dbfefab8748a491,canceled,0,None,None
7,08662c478bed0444d0515925af759547,canceled,0,None,None
8,09f541f117ac3f665dc3aad6d3bb8fc9,canceled,0,None,None
9,0a472c47d928bf4cd8b6a7a30345eeb7,canceled,0,None,None


In [27]:
print(
    "Orders with ZERO item rows:",
    (missing_item_check["order_item_rows"] == 0).sum()
)

print(
    "Orders with item rows:",
    (missing_item_check["order_item_rows"] > 0).sum()
)

Orders with ZERO item rows: 775
Orders with item rows: 0


In [28]:
import pandas as pd
import numpy as np

intervention_path = r"C:\Users\DELL\OneDrive\Desktop\supply-chain-operations-analytics\data\intervention_priority_output.csv"

intervention_df = pd.read_csv(intervention_path)

print("Rows:", len(intervention_df))
print("Columns:", len(intervention_df.columns))

display(intervention_df.head())

Rows: 99441
Columns: 15


,order_id,customer_state,order_item_count,product_count,seller_count,total_order_value,freight_value,late_probability,risk_score,risk_band,impact_score,priority_score,intervention_tier,recommended_action,is_late
0,00010242fe8c5a6d1ba2dd792cb16214,RJ,1.0,1.0,1.0,72.19,13.29,0.507138,50.713810,High,30.324975,42.558276,Standard Intervention,Monitor and review,0
1,00018f77f2f0320c557190d7a144bdd3,SP,1.0,1.0,1.0,259.83,19.93,0.496976,49.697636,Medium,64.913628,55.784033,High Priority,Prioritize fulfillment monitoring,0
2,000229ec398224ef6ca0657da4fc703e,MG,1.0,1.0,1.0,216.87,17.87,0.496846,49.684551,Medium,56.732248,52.503630,High Priority,Prioritize fulfillment monitoring,0
3,00024acbcdf0a6daa1e931b038114c75,SP,1.0,1.0,1.0,25.78,12.79,0.481683,48.168323,Medium,21.640362,37.557139,Standard Intervention,Monitor and review,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,SP,1.0,1.0,1.0,218.04,18.14,0.496670,49.666972,Medium,57.198468,52.679570,High Priority,Prioritize fulfillment monitoring,0


In [29]:
# ============================================================
# STEP 6F — DEFINE VALID OPTIMIZATION POPULATION
# ============================================================

economic_columns = [
    "total_order_value",
    "freight_value",
    "order_item_count",
    "product_count",
    "seller_count"
]

optimization_df = intervention_df.dropna(
    subset=economic_columns
).copy()

print("Original orders:", len(intervention_df))
print("Optimization population:", len(optimization_df))
print(
    "Excluded orders:",
    len(intervention_df) - len(optimization_df)
)

Original orders: 99441
Optimization population: 98666
Excluded orders: 775


In [30]:
print("\nNULL COUNTS:")
print(
    optimization_df[
        economic_columns +
        ["risk_score", "late_probability", "impact_score", "priority_score"]
    ].isna().sum()
)


NULL COUNTS:
total_order_value      0
freight_value          0
order_item_count       0
product_count          0
seller_count           0
risk_score             0
late_probability       0
impact_score         749
priority_score       749
dtype: int64


In [31]:
print("\nROWS:", len(optimization_df))
print(
    "DISTINCT ORDERS:",
    optimization_df["order_id"].nunique()
)


ROWS: 98666
DISTINCT ORDERS: 98666


In [32]:
# ============================================================
# STEP 6H — CALCULATE BUSINESS IMPACT
# ============================================================

optimization_df["impact_score"] = (
    optimization_df["total_order_value"]
)

print("NULL impact scores:",
      optimization_df["impact_score"].isna().sum())

print("\nImpact score statistics:")
print(
    optimization_df["impact_score"].describe()
)

NULL impact scores: 0

Impact score statistics:
count    98666.000000
mean       160.577638
std        220.466087
min          9.590000
25%         61.980000
50%        105.290000
75%        176.870000
max      13664.080000
Name: impact_score, dtype: float64


In [33]:
display(
    optimization_df[
        [
            "order_id",
            "total_order_value",
            "freight_value",
            "impact_score"
        ]
    ].head(10)
)

,order_id,total_order_value,freight_value,impact_score
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,18.14,218.04
5,00048cc3ae777c65dbb7d2a0634bc1ea,34.59,12.69,34.59
6,00054e8431b9d7675808bcb819fb4a32,31.75,11.85,31.75
7,000576fe39319847cbb9d288c5617fa6,880.75,70.75,880.75
8,0005a1a1728c9d785b8e2b08b904576c,157.60,11.65,157.60
9,0005f50442cb953dcd1d21e1fb923495,65.39,11.40,65.39


In [34]:
print(
    optimization_df["impact_score"].describe()
)

count    98666.000000
mean       160.577638
std        220.466087
min          9.590000
25%         61.980000
50%        105.290000
75%        176.870000
max      13664.080000
Name: impact_score, dtype: float64


In [35]:
# ============================================================
# STEP 6I — PRIORITY SCORE
# ============================================================

optimization_df["priority_score"] = (
    optimization_df["risk_score"]
    * optimization_df["impact_score"]
)

print("NULL priority scores:",
      optimization_df["priority_score"].isna().sum())

NULL priority scores: 0


In [36]:
display(
    optimization_df[
        [
            "order_id",
            "late_probability",
            "risk_score",
            "impact_score",
            "priority_score"
        ]
    ]
    .sort_values(
        "priority_score",
        ascending=False
    )
    .head(20)
)

,order_id,late_probability,risk_score,impact_score,priority_score
1471,03caa2c082116e1d31e67e9ae3700499,0.516243,51.624264,13664.08,705398.068752
44830,736e1922ae60d0d6a89247b851902527,0.518340,51.833993,7274.88,377086.081471
99072,fefacc66af859508bf1a7934eab1e97f,0.515952,51.595237,6922.21,357153.062558
3156,0812eb902a67711a1cb742b3cdaa65ae,0.514230,51.422966,6929.31,356325.671307
95187,f5136e38d1a14a4dbd87dff67da82701,0.496568,49.656840,6726.66,334024.679300
17255,2cc9089445046817a7539d90805e6e5a,0.490253,49.025328,6081.54,298149.491638
65576,a96610ab360d42a2e5335a3998b4718a,0.522726,52.272577,4950.34,258767.026572
70098,b4c4b76c642808cbe472a32b86cddc95,0.502508,50.250755,4809.44,241677.992328
54793,8dbc85d1447242f3b127dda390d56e19,0.513354,51.335376,4681.78,240340.937883
9941,199af31afc78c699f0dbf71fb178d4d4,0.497696,49.769618,4764.34,237119.381454


In [37]:
print("Average impact — all candidates:",
      optimization_df["impact_score"].mean())

print("Average impact — top 500:",
      optimization_df.nlargest(
          500, "priority_score"
      )["impact_score"].mean())

print("\nAverage risk — all candidates:",
      optimization_df["risk_score"].mean())

print("Average risk — top 500:",
      optimization_df.nlargest(
          500, "priority_score"
      )["risk_score"].mean())

Average impact — all candidates: 160.57763809214924
Average impact — top 500: 2107.8082000000004

Average risk — all candidates: 49.65223425706334
Average risk — top 500: 50.521509921137664


In [38]:
# ============================================================
# STEP 6K — BASELINE INTERVENTION SELECTION
# ============================================================

INTERVENTION_CAPACITY = 500

baseline_df = (
    optimization_df
    .nlargest(
        INTERVENTION_CAPACITY,
        "priority_score"
    )
    .copy()
)

print("Baseline candidates:", len(baseline_df))
print("Distinct orders:", baseline_df["order_id"].nunique())

print(
    "Late orders:",
    int(baseline_df["is_late"].sum())
)

print(
    "Late rate:",
    round(baseline_df["is_late"].mean() * 100, 2),
    "%"
)

print(
    "Total order value:",
    round(baseline_df["total_order_value"].sum(), 2)
)

print(
    "Average risk score:",
    round(baseline_df["risk_score"].mean(), 4)
)

Baseline candidates: 500
Distinct orders: 500
Late orders: 59
Late rate: 11.8 %
Total order value: 1053904.1
Average risk score: 50.5215


In [39]:
# ============================================================
# STEP 6L — BASELINE CONCENTRATION ANALYSIS
# ============================================================

print("STATE CONCENTRATION")
print("=" * 50)

state_baseline = (
    baseline_df
    .groupby("customer_state")
    .size()
    .sort_values(ascending=False)
)

display(state_baseline.head(15))

print("\nTop state share:")
print(
    round(
        state_baseline.iloc[0] / len(baseline_df) * 100,
        2
    ),
    "%"
)

STATE CONCENTRATION


customer_state
SP    144
RJ     78
MG     50
RS     31
PR     29
BA     26
SC     19
GO     13
PE     13
DF     12
CE     12
PB     11
PA      9
MT      9
MS      8
dtype: int64


Top state share:
28.8 %


In [40]:
print("\nSELLER CONCENTRATION")
print("=" * 50)

# We need seller_id for this analysis.
# Check whether it exists in the intervention dataframe.

print("seller_id exists:",
      "seller_id" in optimization_df.columns)


SELLER CONCENTRATION
seller_id exists: False


In [41]:
# ============================================================
# STEP 6M — CREATE ORDER-SELLER MAPPING
# ============================================================

seller_query = """
SELECT DISTINCT
    order_id,
    seller_id
FROM dbo.raw_order_items
"""

order_seller = pd.read_sql(
    seller_query,
    engine
)

print("Order-seller rows:", len(order_seller))
print("Distinct orders:", order_seller["order_id"].nunique())
print("Distinct sellers:", order_seller["seller_id"].nunique())

display(order_seller.head())

Order-seller rows: 100010
Distinct orders: 98666
Distinct sellers: 3095


,order_id,seller_id
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87


In [42]:
# ============================================================
# STEP 6N — SELLER EXPOSURE IN BASELINE
# ============================================================

baseline_seller = baseline_df[
    ["order_id"]
].merge(
    order_seller,
    on="order_id",
    how="left"
)

seller_exposure = (
    baseline_seller
    .groupby("seller_id")
    .agg(
        intervention_orders=("order_id", "nunique")
    )
    .sort_values(
        "intervention_orders",
        ascending=False
    )
)

print("Distinct sellers involved:",
      seller_exposure.shape[0])

print("\nTop sellers by intervention exposure:")

display(
    seller_exposure.head(20)
)

Distinct sellers involved: 193

Top sellers by intervention exposure:


,intervention_orders
seller_id,
53243585a1d6dc2643021fd1853d8905,35
2bf6a2c1e71bbd29a4ad64e6d3c3629f,17
7e93a43ef30c4f03f38b393420bc753a,15
e882b2a25a10b9c057cc49695f222c19,13
d23019c84ffae2d5ef2270367b8605fc,13
b839e41795b7f3ad94cc2014a52f6796,13
40db9e9aa57f7bb151bcda6b0f9bdbb7,12
ba90964cff9b9e0e6f32b23b82465f7b,12
59417c56835dd8e2e72f91f809cd4092,11


In [43]:
# Maximum seller exposure
max_seller_exposure = seller_exposure[
    "intervention_orders"
].max()

print(
    "Maximum interventions associated with one seller:",
    max_seller_exposure
)

Maximum interventions associated with one seller: 35


In [44]:
# Distribution of seller exposure

print(
    seller_exposure["intervention_orders"].describe()
)

count    193.000000
mean       2.632124
std        3.603298
min        1.000000
25%        1.000000
50%        1.000000
75%        3.000000
max       35.000000
Name: intervention_orders, dtype: float64


In [45]:
SELLER_CAP = 5

seller_violations = (
    seller_exposure[
        "intervention_orders"
    ] > SELLER_CAP
)

print(
    "Sellers exceeding cap:",
    seller_violations.sum()
)

print(
    "Orders associated with over-cap sellers:",
    seller_exposure.loc[
        seller_violations,
        "intervention_orders"
    ].sum()
)

Sellers exceeding cap: 16
Orders associated with over-cap sellers: 193


In [46]:
# ============================================================
# STEP 6P — STATE CONSTRAINT ANALYSIS
# ============================================================

STATE_CAP = 100

state_exposure = (
    baseline_df
    .groupby("customer_state")
    .size()
    .sort_values(ascending=False)
)

state_violations = (
    state_exposure > STATE_CAP
)

print(
    "States exceeding cap:",
    state_violations.sum()
)

print("\nStates exceeding cap:")

display(
    state_exposure[
        state_violations
    ]
)

States exceeding cap: 1

States exceeding cap:


customer_state
SP    144
dtype: int64

In [47]:
# ============================================================
# STEP 6R — SELLER CAP FEASIBILITY
# ============================================================

print("Total optimization candidates:",
      len(optimization_df))

print("Distinct sellers:",
      order_seller["seller_id"].nunique())

print("Maximum baseline seller exposure:",
      seller_exposure["intervention_orders"].max())

print("Sellers with >5 baseline interventions:",
      (seller_exposure["intervention_orders"] > 5).sum())

print("\nSeller exposure distribution:")

display(
    seller_exposure[
        "intervention_orders"
    ].value_counts()
    .sort_index()
)

Total optimization candidates: 98666
Distinct sellers: 3095
Maximum baseline seller exposure: 35
Sellers with >5 baseline interventions: 16

Seller exposure distribution:


intervention_orders
1     98
2     43
3     19
4     11
5      6
6      3
7      2
9      1
11     2
12     2
13     3
15     1
17     1
35     1
Name: count, dtype: int64

In [48]:
# ============================================================
# STEP 6R.1 — ORDERS WITH HIGH-EXPOSURE SELLERS
# ============================================================

high_exposure_sellers = set(
    seller_exposure[
        seller_exposure["intervention_orders"] > 5
    ].index
)

candidate_seller_map = (
    order_seller[
        order_seller["seller_id"].isin(
            high_exposure_sellers
        )
    ]
)

affected_candidate_orders = (
    candidate_seller_map["order_id"]
    .nunique()
)

print(
    "Candidate orders associated with high-exposure sellers:",
    affected_candidate_orders
)

Candidate orders associated with high-exposure sellers: 1261


In [49]:
# ============================================================
# STEP 6S.1 — PREPARE OPTIMIZATION DATA
# ============================================================

import pandas as pd
import numpy as np

# Work on a clean copy
opt = optimization_df.copy()

# Make sure order_id is unique
assert opt["order_id"].is_unique, \
    "Optimization dataset is not at order-level grain."

# Remove any accidental null priority values
opt = opt.dropna(
    subset=[
        "priority_score",
        "risk_score",
        "impact_score",
        "customer_state"
    ]
).copy()

print("Optimization rows:", len(opt))
print("Distinct orders:", opt["order_id"].nunique())
print("NULL priority scores:", opt["priority_score"].isna().sum())

Optimization rows: 98666
Distinct orders: 98666
NULL priority scores: 0


In [50]:
# ============================================================
# STEP 6S.2 — CREATE CANDIDATE INDEX
# ============================================================

opt = opt.reset_index(drop=True)

opt["candidate_id"] = np.arange(len(opt))

print(
    opt[
        [
            "candidate_id",
            "order_id",
            "customer_state",
            "risk_score",
            "impact_score",
            "priority_score"
        ]
    ].head()
)

   candidate_id                          order_id customer_state  risk_score  \
0             0  00010242fe8c5a6d1ba2dd792cb16214             RJ   50.713810   
1             1  00018f77f2f0320c557190d7a144bdd3             SP   49.697636   
2             2  000229ec398224ef6ca0657da4fc703e             MG   49.684551   
3             3  00024acbcdf0a6daa1e931b038114c75             SP   48.168323   
4             4  00042b26cf59d7ce69dfabb4e55b4fd9             SP   49.666972   

   impact_score  priority_score  
0         72.19     3661.029927  
1        259.83    12912.936772  
2        216.87    10775.088575  
3         25.78     1241.779364  
4        218.04    10829.386546  


In [51]:
# ============================================================
# STEP 6S.3 — CANDIDATE ORDER-SELLER MAPPING
# ============================================================

candidate_ids = set(opt["order_id"])

candidate_order_seller = (
    order_seller[
        order_seller["order_id"].isin(candidate_ids)
    ]
    .drop_duplicates()
)

print(
    "Candidate order-seller relationships:",
    len(candidate_order_seller)
)

print(
    "Candidate orders with seller mapping:",
    candidate_order_seller["order_id"].nunique()
)

print(
    "Candidate sellers:",
    candidate_order_seller["seller_id"].nunique()
)

Candidate order-seller relationships: 100010
Candidate orders with seller mapping: 98666
Candidate sellers: 3095


In [52]:
# ============================================================
# STEP 6S.4 — CONSTRAINED INTERVENTION OPTIMIZATION
# ============================================================

from pulp import (
    LpProblem,
    LpMaximize,
    LpVariable,
    lpSum,
    LpBinary,
    LpStatus
)

CAPACITY = 500
STATE_CAP = 100
SELLER_CAP = 5

model = LpProblem(
    "Intervention_Prioritization",
    LpMaximize
)

# Binary decision variable:
# x[i] = 1 → select order
# x[i] = 0 → don't select order

x = {
    i: LpVariable(
        f"x_{i}",
        cat=LpBinary
    )
    for i in opt["candidate_id"]
}

In [53]:
# ============================================================
# STEP 6S.5 — OBJECTIVE
# ============================================================

priority_lookup = (
    opt
    .set_index("candidate_id")["priority_score"]
    .to_dict()
)

model += lpSum(
    priority_lookup[i] * x[i]
    for i in x
)

print("Objective created.")

Objective created.


In [54]:
# ============================================================
# STEP 6S.6 — CAPACITY CONSTRAINT
# ============================================================

model += (
    lpSum(x[i] for i in x)
    <= CAPACITY
)

print("Capacity constraint added.")

Capacity constraint added.


In [55]:
# ============================================================
# STEP 6S.7 — STATE CONSTRAINTS
# ============================================================

for state in opt["customer_state"].dropna().unique():

    state_candidates = opt.loc[
        opt["customer_state"] == state,
        "candidate_id"
    ]

    model += (
        lpSum(x[i] for i in state_candidates)
        <= STATE_CAP
    )

print(
    "State constraints added:",
    opt["customer_state"].nunique()
)

State constraints added: 27


In [56]:
# ============================================================
# STEP 6S.8 — SELLER EXPOSURE CONSTRAINTS
# ============================================================

order_to_candidate = (
    opt
    .set_index("order_id")["candidate_id"]
    .to_dict()
)

seller_constraint_count = 0

for seller, group in candidate_order_seller.groupby("seller_id"):

    candidate_variables = []

    for order_id in group["order_id"].unique():

        if order_id in order_to_candidate:

            candidate_variables.append(
                x[order_to_candidate[order_id]]
            )

    if candidate_variables:

        model += (
            lpSum(candidate_variables)
            <= SELLER_CAP
        )

        seller_constraint_count += 1

print(
    "Seller constraints added:",
    seller_constraint_count
)

Seller constraints added: 3095


In [57]:
# ============================================================
# STEP 6S.9 — SOLVE
# ============================================================

print("Starting optimization...")

model.solve()

print(
    "Solver status:",
    LpStatus[model.status]
)

Starting optimization...
Solver status: Optimal


In [58]:
# ============================================================
# STEP 6S.10 — EXTRACT OPTIMIZED PORTFOLIO
# ============================================================

selected_candidate_ids = [
    i
    for i in x
    if x[i].value() == 1
]

optimized_df = opt[
    opt["candidate_id"].isin(
        selected_candidate_ids
    )
].copy()

print(
    "Selected orders:",
    len(optimized_df)
)

print(
    "Distinct orders:",
    optimized_df["order_id"].nunique()
)

Selected orders: 500
Distinct orders: 500


In [59]:
# ============================================================
# STEP 6T — BASELINE VS OPTIMIZED PORTFOLIO
# ============================================================

# Baseline portfolio should already exist
# If your variable is named differently, use that existing
# baseline Top-500 dataframe.

print("Baseline rows:", len(baseline_df))
print("Optimized rows:", len(optimized_df))

print(
    "Baseline distinct orders:",
    baseline_df["order_id"].nunique()
)

print(
    "Optimized distinct orders:",
    optimized_df["order_id"].nunique()
)

Baseline rows: 500
Optimized rows: 500
Baseline distinct orders: 500
Optimized distinct orders: 500


In [60]:
# ============================================================
# STEP 6T.1 — PORTFOLIO KPI COMPARISON
# ============================================================

comparison = pd.DataFrame({
    "Metric": [
        "Selected Orders",
        "Late Orders",
        "Late Rate (%)",
        "Total Order Value",
        "Average Order Value",
        "Average Risk Score",
        "Average Impact Score",
        "Average Priority Score"
    ],

    "Baseline Top-500": [
        len(baseline_df),

        int(baseline_df["is_late"].sum()),

        baseline_df["is_late"].mean() * 100,

        baseline_df["total_order_value"].sum(),

        baseline_df["total_order_value"].mean(),

        baseline_df["risk_score"].mean(),

        baseline_df["impact_score"].mean(),

        baseline_df["priority_score"].mean()
    ],

    "Optimized 500": [
        len(optimized_df),

        int(optimized_df["is_late"].sum()),

        optimized_df["is_late"].mean() * 100,

        optimized_df["total_order_value"].sum(),

        optimized_df["total_order_value"].mean(),

        optimized_df["risk_score"].mean(),

        optimized_df["impact_score"].mean(),

        optimized_df["priority_score"].mean()
    ]
})

display(comparison)

,Metric,Baseline Top-500,Optimized 500
0,Selected Orders,5.000000e+02,5.000000e+02
1,Late Orders,5.900000e+01,5.500000e+01
2,Late Rate (%),1.180000e+01,1.100000e+01
3,Total Order Value,1.053904e+06,1.003394e+06
4,Average Order Value,2.107808e+03,2.006787e+03
5,Average Risk Score,5.052151e+01,5.061969e+01
6,Average Impact Score,2.107808e+03,2.006787e+03
7,Average Priority Score,1.065044e+05,1.015533e+05


In [61]:
# ============================================================
# STEP 6T.2 — OPTIMIZATION IMPROVEMENT
# ============================================================

baseline_late_rate = baseline_df["is_late"].mean() * 100
optimized_late_rate = optimized_df["is_late"].mean() * 100

baseline_late_orders = baseline_df["is_late"].sum()
optimized_late_orders = optimized_df["is_late"].sum()

baseline_value = baseline_df["total_order_value"].sum()
optimized_value = optimized_df["total_order_value"].sum()

baseline_impact = baseline_df["impact_score"].mean()
optimized_impact = optimized_df["impact_score"].mean()

print("LATE ORDER CHANGE:",
      optimized_late_orders - baseline_late_orders)

print(
    "LATE RATE CHANGE:",
    optimized_late_rate - baseline_late_rate,
    "percentage points"
)

print(
    "ORDER VALUE CHANGE:",
    optimized_value - baseline_value
)

print(
    "AVERAGE IMPACT CHANGE:",
    optimized_impact - baseline_impact
)

LATE ORDER CHANGE: -4
LATE RATE CHANGE: -0.7999999999999989 percentage points
ORDER VALUE CHANGE: -50510.35000000009
AVERAGE IMPACT CHANGE: -101.02070000000049


In [62]:
# ============================================================
# STEP 6U — STATE CONSTRAINT VALIDATION
# ============================================================

optimized_state_exposure = (
    optimized_df
    .groupby("customer_state")
    .size()
    .sort_values(ascending=False)
)

print("Maximum state exposure:",
      optimized_state_exposure.max())

print("\nTop states:")

display(
    optimized_state_exposure.head(15)
)

print(
    "\nStates exceeding cap:",
    (optimized_state_exposure > STATE_CAP).sum()
)

Maximum state exposure: 100

Top states:


customer_state
SP    100
RJ     91
MG     50
RS     33
PR     32
BA     32
SC     21
GO     16
PB     13
PE     13
PA     12
DF     12
MS     11
MT     11
CE     11
dtype: int64


States exceeding cap: 0


In [63]:
# ============================================================
# STEP 6U.1 — SELLER CONSTRAINT VALIDATION
# ============================================================

selected_orders = set(
    optimized_df["order_id"]
)

optimized_seller_exposure = (
    candidate_order_seller[
        candidate_order_seller["order_id"]
        .isin(selected_orders)
    ]
    .drop_duplicates()
    .groupby("seller_id")
    .size()
    .sort_values(ascending=False)
)

print(
    "Maximum seller exposure:",
    optimized_seller_exposure.max()
)

print("\nTop sellers:")

display(
    optimized_seller_exposure.head(20)
)

print(
    "\nSellers exceeding cap:",
    (optimized_seller_exposure > SELLER_CAP).sum()
)

Maximum seller exposure: 5

Top sellers:


seller_id
039e6ad9dae79614493083e241147386    5
05feb94f19d094d4b0f9281f0b1d4c99    5
04308b1ee57b6625f47df1d56f00eedf    5
06532f10282704ef4c69168b914b77be    5
17f51e7198701186712e53a39c564617    5
25c5c91f63607446a97b143d2d535d31    5
40db9e9aa57f7bb151bcda6b0f9bdbb7    5
397c4d0c005b6f41f90098ac724e28cb    5
33dd941c27854f7625b968cc6195a552    5
2bf6a2c1e71bbd29a4ad64e6d3c3629f    5
2b3e4a2a3ea8e01938cabda2a3e5cc79    5
54219883e72aad869adfb2a54b7bfa0f    5
53243585a1d6dc2643021fd1853d8905    5
52f976b17ea7f2f087f56dcc419328f6    5
5d378b73ab7dd6f0418d743e5dcb0bd1    5
6061155addc1e54b4cfb51c1c2a32ad8    5
59417c56835dd8e2e72f91f809cd4092    5
eeb6de78f79159600292e314a77cbd18    5
e882b2a25a10b9c057cc49695f222c19    5
b839e41795b7f3ad94cc2014a52f6796    5
dtype: int64


Sellers exceeding cap: 0


In [64]:
# ============================================================
# STEP 6V — INTERVENTION COVERAGE
# ============================================================

total_late_orders = optimization_df["is_late"].sum()

baseline_late_capture = (
    baseline_df["is_late"].sum()
)

optimized_late_capture = (
    optimized_df["is_late"].sum()
)

baseline_coverage = (
    baseline_late_capture /
    total_late_orders * 100
)

optimized_coverage = (
    optimized_late_capture /
    total_late_orders * 100
)

print(
    "Total late orders:",
    total_late_orders
)

print(
    "Baseline late-order capture:",
    baseline_late_capture
)

print(
    "Optimized late-order capture:",
    optimized_late_capture
)

print(
    "Baseline coverage:",
    baseline_coverage,
    "%"
)

print(
    "Optimized coverage:",
    optimized_coverage,
    "%"
)

Total late orders: 7827
Baseline late-order capture: 59
Optimized late-order capture: 55
Baseline coverage: 0.7538009454452537 %
Optimized coverage: 0.7026957966015076 %


In [65]:
# ============================================================
# STEP 6W — FINAL OPTIMIZATION SUMMARY
# ============================================================

optimization_summary = {
    "candidate_orders": len(optimization_df),

    "intervention_capacity": CAPACITY,

    "selected_orders": len(optimized_df),

    "baseline_late_orders": int(
        baseline_df["is_late"].sum()
    ),

    "optimized_late_orders": int(
        optimized_df["is_late"].sum()
    ),

    "baseline_late_rate": round(
        baseline_df["is_late"].mean() * 100,
        2
    ),

    "optimized_late_rate": round(
        optimized_df["is_late"].mean() * 100,
        2
    ),

    "baseline_total_value": round(
        baseline_df["total_order_value"].sum(),
        2
    ),

    "optimized_total_value": round(
        optimized_df["total_order_value"].sum(),
        2
    ),

    "baseline_avg_risk": round(
        baseline_df["risk_score"].mean(),
        2
    ),

    "optimized_avg_risk": round(
        optimized_df["risk_score"].mean(),
        2
    ),

    "baseline_avg_impact": round(
        baseline_df["impact_score"].mean(),
        2
    ),

    "optimized_avg_impact": round(
        optimized_df["impact_score"].mean(),
        2
    ),

    "state_cap": STATE_CAP,

    "seller_cap": SELLER_CAP
}

display(
    pd.DataFrame([optimization_summary])
)

,candidate_orders,intervention_capacity,selected_orders,baseline_late_orders,optimized_late_orders,baseline_late_rate,optimized_late_rate,baseline_total_value,optimized_total_value,baseline_avg_risk,optimized_avg_risk,baseline_avg_impact,optimized_avg_impact,state_cap,seller_cap
0,98666,500,500,59,55,11.8,11.0,1053904.1,1003393.75,50.52,50.62,2107.81,2006.79,100,5


In [66]:
# ============================================================
# FINAL INTERVENTION OPTIMIZATION EVALUATION
# ============================================================

print("=" * 70)
print("INTERVENTION OPTIMIZATION RESULTS")
print("=" * 70)

# ------------------------------------------------------------
# 1. BASIC VALIDATION
# ------------------------------------------------------------

print("\n1. PORTFOLIO SIZE")
print("-" * 50)

print("Optimization population:", len(optimization_df))
print("Baseline portfolio:", len(baseline_df))
print("Optimized portfolio:", len(optimized_df))

assert len(optimized_df) == CAPACITY
assert optimized_df["order_id"].nunique() == CAPACITY

print("✓ Capacity constraint satisfied")


# ------------------------------------------------------------
# 2. STATE CONSTRAINT
# ------------------------------------------------------------

optimized_state_exposure = (
    optimized_df
    .groupby("customer_state")
    .size()
    .sort_values(ascending=False)
)

state_violations = (
    optimized_state_exposure > STATE_CAP
).sum()

print("\n2. STATE CONSTRAINT")
print("-" * 50)

print("Maximum state exposure:",
      optimized_state_exposure.max())

print("States exceeding cap:",
      state_violations)

assert state_violations == 0

print("✓ State constraint satisfied")


# ------------------------------------------------------------
# 3. SELLER CONSTRAINT
# ------------------------------------------------------------

selected_orders = set(
    optimized_df["order_id"]
)

optimized_seller_exposure = (
    candidate_order_seller[
        candidate_order_seller["order_id"]
        .isin(selected_orders)
    ]
    .drop_duplicates()
    .groupby("seller_id")
    .size()
    .sort_values(ascending=False)
)

seller_violations = (
    optimized_seller_exposure > SELLER_CAP
).sum()

print("\n3. SELLER CONSTRAINT")
print("-" * 50)

print("Maximum seller exposure:",
      optimized_seller_exposure.max())

print("Sellers exceeding cap:",
      seller_violations)

assert seller_violations == 0

print("✓ Seller constraint satisfied")


# ------------------------------------------------------------
# 4. PORTFOLIO PERFORMANCE
# ------------------------------------------------------------

baseline_late = int(
    baseline_df["is_late"].sum()
)

optimized_late = int(
    optimized_df["is_late"].sum()
)

total_late = int(
    optimization_df["is_late"].sum()
)

baseline_late_rate = (
    baseline_df["is_late"].mean() * 100
)

optimized_late_rate = (
    optimized_df["is_late"].mean() * 100
)

baseline_coverage = (
    baseline_late / total_late * 100
)

optimized_coverage = (
    optimized_late / total_late * 100
)


# ------------------------------------------------------------
# 5. VALUE / RISK / IMPACT
# ------------------------------------------------------------

baseline_value = (
    baseline_df["total_order_value"].sum()
)

optimized_value = (
    optimized_df["total_order_value"].sum()
)

baseline_risk = (
    baseline_df["risk_score"].mean()
)

optimized_risk = (
    optimized_df["risk_score"].mean()
)

baseline_impact = (
    baseline_df["impact_score"].mean()
)

optimized_impact = (
    optimized_df["impact_score"].mean()
)


# ------------------------------------------------------------
# 6. FINAL COMPARISON
# ------------------------------------------------------------

comparison = pd.DataFrame({

    "Metric": [
        "Selected Orders",
        "Late Orders",
        "Late Rate (%)",
        "Late Order Coverage (%)",
        "Total Order Value",
        "Average Risk Score",
        "Average Impact Score"
    ],

    "Baseline Top-500": [

        len(baseline_df),

        baseline_late,

        round(baseline_late_rate, 2),

        round(baseline_coverage, 2),

        round(baseline_value, 2),

        round(baseline_risk, 2),

        round(baseline_impact, 2)
    ],

    "Optimized 500": [

        len(optimized_df),

        optimized_late,

        round(optimized_late_rate, 2),

        round(optimized_coverage, 2),

        round(optimized_value, 2),

        round(optimized_risk, 2),

        round(optimized_impact, 2)
    ]
})

print("\n4. BASELINE VS OPTIMIZED")
print("-" * 50)

display(comparison)


# ------------------------------------------------------------
# 7. IMPROVEMENT
# ------------------------------------------------------------

print("\n5. OPTIMIZATION IMPACT")
print("-" * 50)

print(
    "Late orders captured:",
    optimized_late
)

print(
    "Late-order coverage:",
    round(optimized_coverage, 2),
    "%"
)

print(
    "Average impact:",
    round(optimized_impact, 2)
)

print(
    "Average risk:",
    round(optimized_risk, 2)
)


# ------------------------------------------------------------
# 8. SAVE FINAL DATASET
# ------------------------------------------------------------

final_output = optimized_df.copy()

output_path = (
    r"C:\Users\DELL\OneDrive\Desktop"
    r"\supply-chain-operations-analytics"
    r"\data\intervention_priority"
    r"\final_optimized_interventions.csv"
)

import os

os.makedirs(
    os.path.dirname(output_path),
    exist_ok=True
)

final_output.to_csv(
    output_path,
    index=False
)

print("\n6. OUTPUT")
print("-" * 50)

print(
    "Saved:",
    output_path
)

print("\n✓ INTERVENTION OPTIMIZATION COMPLETE")

INTERVENTION OPTIMIZATION RESULTS

1. PORTFOLIO SIZE
--------------------------------------------------
Optimization population: 98666
Baseline portfolio: 500
Optimized portfolio: 500
✓ Capacity constraint satisfied

2. STATE CONSTRAINT
--------------------------------------------------
Maximum state exposure: 100
States exceeding cap: 0
✓ State constraint satisfied

3. SELLER CONSTRAINT
--------------------------------------------------
Maximum seller exposure: 5
Sellers exceeding cap: 0
✓ Seller constraint satisfied

4. BASELINE VS OPTIMIZED
--------------------------------------------------


,Metric,Baseline Top-500,Optimized 500
0,Selected Orders,500.00,500.00
1,Late Orders,59.00,55.00
2,Late Rate (%),11.80,11.00
3,Late Order Coverage (%),0.75,0.70
4,Total Order Value,1053904.10,1003393.75
5,Average Risk Score,50.52,50.62
6,Average Impact Score,2107.81,2006.79



5. OPTIMIZATION IMPACT
--------------------------------------------------
Late orders captured: 55
Late-order coverage: 0.7 %
Average impact: 2006.79
Average risk: 50.62

6. OUTPUT
--------------------------------------------------
Saved: C:\Users\DELL\OneDrive\Desktop\supply-chain-operations-analytics\data\intervention_priority\final_optimized_interventions.csv

✓ INTERVENTION OPTIMIZATION COMPLETE


In [67]:
query = """
SELECT *
FROM dbo.order_analytics
"""

order_analytics_export = pd.read_sql(query, engine)

print("Rows:", len(order_analytics_export))
print("Columns:", len(order_analytics_export.columns))

order_analytics_export.to_csv( r"C:\Users\DELL\OneDrive\Desktop\supply-chain-operations-analytics\data\order_analytics.csv",
    index=False
)

Rows: 99441
Columns: 40
